# 분석 개요

1. 유저의 지점 이용행태 분석을 통한 환불 원인 파악
2. 프로모션구매 기록 분석을 통한 환불 원인 파악
3. 그외 환불에 영향을 미치는 요인 분석 (웹/앱 행동기반)
4. 유저가 선호 지점이 없어서 환불한걸 확인할 방법이있을지?


# DB 데이터 SQL

#계약시작전 환불(WITHDRAW, CANCELED)

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_payment = pd.read_sql("""
                        SELECT

                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS p_date,
                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH:MM:SS') AS p_time,
                        c.contract_uid transaction_id,
                        c.client_uid uid,
                        c.status,
                        ph.payment_status AS ph_status,
                        c.product_name,
                        pp.name product_period,
                        TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                        TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                        TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                        c.actual_price,
                        TO_CHAR(c.created_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS created_at,
                        c.is_migrated,
                        p.order_id,
                        u.phone_number
                        FROM contract c
                        LEFT JOIN payment p
                        ON c.contract_uid = p.contract_payment_uid
                        LEFT JOIN price_policy pp
                        ON c.price_policy_uid = pp.price_policy_uid
                        LEFT JOIN client u
                        ON c.client_uid = u.client_uid
                        LEFT JOIN payment_history ph
                        ON contract_uid = ph.payment_uid
                        where actual_price > 0
                        ORDER BY ph.requested_at ASC;

                    """, conn)
    conn.close()
df_payment['product_name'] = df_payment['product_name'] + ' ' + df_payment['product_period'].astype(str)
df_payment = df_payment.drop(columns=['product_period'])
df_payment.tail()

In [ ]:
df_payment.groupby('status')['uid'].nunique().reset_index()

In [ ]:
# 1. 분류 로직 함수 정의
def categorize(status_series):
    # 해당 유저(그룹)의 모든 status를 집합으로 변환
    statuses = set(status_series)

    # 우선순위에 따른 조건문
    if 'TERMINATION' in statuses:
        return '계약시작후환불'
    elif 'CANCELED' in statuses or 'WITHDRAW' in statuses:
        return '계약시작전환불'
    else:
        return '정상이용유저'

# 2. groupby().transform()을 사용하여 새로운 컬럼 추가
# 이렇게 하면 기존 df_payment의 행 순서를 유지하면서 유저별 결과가 채워집니다.
df_payment['user_category'] = df_payment.groupby('phone_number')['status'].transform(categorize)

df_payment.head()

In [ ]:
df_payment.groupby('user_category')['actual_price'].sum().reset_index()

In [ ]:
import pandas as pd

# 1. p_date를 datetime 형식으로 변환 (필수)
df_payment['p_date'] = pd.to_datetime(df_payment['p_date'])

# 2. 기간 필터링 (2025-06-11 ~ 2026-04-30)
# 시작일과 종료일을 설정합니다.
start_date = '2025-06-11'
end_date = '2026-04-30'

# 해당 기간에 포함되는 데이터만 추출합니다.
df_payment = df_payment[(df_payment['p_date'] >= start_date) & (df_payment['p_date'] <= end_date)]

# 3. 필터링 결과 확인 (정렬까지 포함하면 더 보기 좋습니다)
df_payment = df_payment.sort_values('p_date')
df_payment.head()

#프로모션별 성과 분석

In [ ]:
import pandas as pd

# 1. 프로모션 데이터 정의 (2025년 ~ 2026년 전체)
promo_intervals = [
    # --- 2025년 ---
    ('2025-06-12', '2025-06-22', 'Launch_Festa_2025'),
    ('2025-06-23', '2025-06-29', 'June_Last_Call_Sale'),
    ('2025-06-30', '2025-07-06', 'July_Hot_Deal_Promo'),
    ('2025-07-07', '2025-07-13', 'Flex_Week_Q3_01'),
    ('2025-07-14', '2025-07-21', 'Q3_Flash_Time_Attack'),
    ('2025-07-22', '2025-07-28', 'Summer_Mega_Campaign'),
    ('2025-07-29', '2025-08-04', 'Level_Up_Project_V1'),
    ('2025-08-05', '2025-08-11', 'Value_Boost_Promotion'),
    ('2025-08-12', '2025-08-18', 'August_Growth_Special'),
    ('2025-08-19', '2025-08-25', 'Flex_Week_Q3_02'),
    ('2025-08-26', '2025-09-01', 'Level_Up_Project_V2'),
    ('2025-09-02', '2025-09-08', 'Autumn_Kickoff_Sale'),
    ('2025-09-09', '2025-09-15', 'Autumn_Flash_Week'),
    ('2025-09-16', '2025-09-22', 'Q3_Countdown_Special'),
    ('2025-09-23', '2025-09-29', 'Brand_Focus_Campaign'),
    ('2025-09-30', '2025-10-02', 'First_Come_Coupon_Run'),
    ('2025-10-03', '2025-10-13', 'Spot_Onboarding_Drive'),
    ('2025-10-14', '2025-10-20', 'October_Last_Spurt'),
    ('2025-10-21', '2025-10-27', 'Miracle_Season_Sale_V1'),
    ('2025-10-28', '2025-11-03', 'Retention_Support_Subsidy'),
    ('2025-11-04', '2025-11-10', 'Daily_Habit_Challenge'),
    ('2025-11-11', '2025-11-17', 'Best_Price_Guarantee_Q4'),
    ('2025-11-18', '2025-12-01', 'Black_Friday_Mega_Sale'),
    ('2025-12-02', '2025-12-08', '4th_Anniversary_Festival'),
    ('2025-12-09', '2025-12-15', 'Winter_Golden_Week'),
    ('2025-12-16', '2025-12-31', 'Year_End_Grand_Sale'),
    # --- 2026년 ---
    ('2026-01-01', '2026-01-07', 'New_Year_Festa_2026'),
    ('2026-01-08', '2026-01-14', 'New_Year_Resolution_Promo'),
    ('2026-01-15', '2026-01-21', 'Q1_Jump_Up_Campaign'),
    ('2026-01-22', '2026-01-28', 'January_Closing_Deal'),
    ('2026-01-29', '2026-02-04', 'Winter_Closing_Festival'),
    ('2026-02-05', '2026-02-11', 'Lunar_New_Year_Special'),
    ('2026-02-12', '2026-02-18', 'First_Come_Lunar_Sale'),
    ('2026-02-19', '2026-02-25', 'Double_Benefit_Week'),
    ('2026-02-26', '2026-03-04', 'Spring_Early_Bird_Sale'),
    ('2026-03-05', '2026-03-11', 'Spring_Jump_Campaign'),
    ('2026-03-12', '2026-03-18', 'Best_Price_Guarantee_Q1'),
    ('2026-03-19', '2026-03-25', 'Loyalty_Thank_You_Kit'),
    ('2026-03-26', '2026-04-01', 'April_Fools_Day_Special'),
    ('2026-04-02', '2026-04-08', 'Miracle_Season_Sale_V2'),
    ('2026-04-09', '2026-04-15', 'One_Day_Lucky_Sale'),
    ('2026-04-16', '2026-04-22', 'Cash_Back_Bundle_Week'),
    ('2026-04-23', '2026-04-30', 'Buy_1_Get_1_Bonus_Week'),
]

# 2. 프로모션 정렬 및 탐색용 룩업 데이터프레임 구조화
df_promo_lookup = pd.DataFrame(promo_intervals, columns=['프로모션_시작일', '프로모션_종료일', '프로모션'])
df_promo_lookup['프로모션_시작일'] = pd.to_datetime(df_promo_lookup['프로모션_시작일'])
df_promo_lookup['프로모션_종료일'] = pd.to_datetime(df_promo_lookup['프로모션_종료일'])

# 3. 데이터 타입 정렬 (merge_asof 선행 조건: 시계열 정렬)
df_payment['p_date'] = pd.to_datetime(df_payment['p_date'])
df_payment = df_payment.sort_values('p_date')
df_promo_lookup = df_promo_lookup.sort_values('프로모션_시작일')

# 4. row-by-row loop를 배제한 pd.merge_asof 기반 고속 범위 조인 (O(N log M) 벡터 연산)
df_payment = pd.merge_asof(
    df_payment, df_promo_lookup,
    left_on='p_date', right_on='프로모션_시작일',
    direction='backward'
)

# 5. 시작일 기준 매핑 후, 실제 종료일을 벗어난 예외 데이터 필터링 및 예외 처리
is_outside_interval = df_payment['p_date'] > df_payment['프로모션_종료일']
df_payment.loc[is_outside_interval, ['프로모션', '프로모션_시작일', '프로모션_종료일']] = [None, pd.NaT, pd.NaT]
df_payment['프로모션'] = df_payment['프로모션'].fillna('프로모션 없음')

# 결과 확인
df_payment.head()

In [ ]:
import re

def clean_product_name(name):
    if not isinstance(name, str):
        return name

    # 1. 기본 명칭 통일 (무제한 -> 무제한 패스, 주말&야간 -> 주말&야간 패스)
    # 이미 '패스'가 포함되어 있으면 중복해서 붙이지 않음
    if '무제한' in name and '패스' not in name:
        name = name.replace('무제한', '무제한 패스')
    if '주말&야간' in name and '패스' not in name:
        name = name.replace('주말&야간', '주말&야간 패스')

    # 2. 산술 연산 처리 (6x2 -> 12, 6+6 -> 12, 12+1 -> 13)
    # 곱셈 패턴 처리 (숫자 x 숫자)
    mult_match = re.search(r'(\d+)\s*[xX]\s*(\d+)', name)
    if mult_match:
        calc_val = int(mult_match.group(1)) * int(mult_match.group(2))
        name = re.sub(r'\d+\s*[xX]\s*\d+', str(calc_val), name)

    # 덧셈 패턴 처리 (숫자 + 숫자)
    plus_match = re.search(r'(\d+)\s*\+\s*(\d+)', name)
    if plus_match:
        calc_val = int(plus_match.group(1)) + int(plus_match.group(2))
        name = re.sub(r'\d+\s*\+\s*\d+', str(calc_val), name)

    # 3. 기타 불필요한 기호 및 공백 정리
    name = name.replace('+', ' ') # 숫자가 아닌 곳에 붙은 + 제거
    name = re.sub(r'\s+', ' ', name) # 중복 공백을 하나로 합침

    return name.strip()

# 데이터프레임 적용 예시
df_payment['product_name_clean'] = df_payment['product_name'].apply(clean_product_name)

In [ ]:
# 1. 결제 완료(DONE) 데이터에서 transaction_id별 프로모션 매핑 테이블 생성
# 동일한 transaction_id에 대해 결제 시점의 프로모션명을 가져옵니다.
done_promo_mapping = df_payment[df_payment['ph_status'] == 'DONE'][['transaction_id', '프로모션']].drop_duplicates('transaction_id')
done_promo_mapping.columns = ['transaction_id', 'temp_결제시점_프로모션']

# 2. 전체 데이터프레임에 매핑 테이블 병합(Merge)
df_payment = df_payment.merge(done_promo_mapping, on='transaction_id', how='left')

# 3. '프로모션_결제시점' 컬럼 생성
# CANCELED인 경우 매핑된 결제 시점 프로모션을 사용하고,
# 매핑 값이 없거나 DONE인 경우 현재 행의 프로모션을 사용합니다.
df_payment['프로모션_결제시점'] = df_payment['temp_결제시점_프로모션'].fillna(df_payment['프로모션'])

# 4. 임시 컬럼 삭제 및 확인
df_payment.drop(columns=['temp_결제시점_프로모션'], inplace=True)

# 잘 반영되었는지 확인 (취소건 위주로 확인해보세요)
display(df_payment[df_payment['ph_status'] == 'CANCELED'][['p_date', 'ph_status', '프로모션', '프로모션_결제시점']].head())

In [ ]:
# 1. 결제 완료(DONE) 시점의 상세 프로모션 정보(이름, 시작일, 종료일) 매핑 테이블 생성
# transaction_id를 기준으로 실제 결제 당시의 모든 프로모션 정보를 가져옵니다.
done_info_mapping = df_payment[df_payment['ph_status'] == 'DONE'][
    ['transaction_id', '프로모션', '프로모션_시작일', '프로모션_종료일']
].drop_duplicates('transaction_id')

# 컬럼명 중복 방지를 위해 이름 변경
done_info_mapping.columns = ['transaction_id', '결제시점_프로모션', '결제시점_시작일', '결제시점_종료일']

# 2. 전체 데이터(df_payment)에 결제 시점 정보 병합
# 기존에 작업하신 df_payment에 해당 정보를 붙입니다.
df_payment = df_payment.merge(done_info_mapping, on='transaction_id', how='left')

# 결제 시점 정보가 없는 경우(예: 결제 데이터가 누락된 취소건 등) 현재 행의 정보를 유지합니다.
df_payment['결제시점_프로모션'] = df_payment['결제시점_프로모션'].fillna(df_payment['프로모션'])
df_payment['결제시점_시작일'] = df_payment['결제시점_시작일'].fillna(df_payment['프로모션_시작일'])
df_payment['결제시점_종료일'] = df_payment['결제시점_종료일'].fillna(df_payment['프로모션_종료일'])

# 3. 결제 시점 프로모션을 기준으로 그룹화하여 집계
# ph_status에 상관없이 모든 행이 '실제 결제된 시점'의 프로모션으로 모입니다.
total_promotion_stats = df_payment.groupby(
    ['user_category', '결제시점_시작일', '결제시점_종료일', '결제시점_프로모션', 'product_name']
).agg(
    구매건수=('ph_status', lambda x: (x == 'DONE').sum()),
    구매총액=('actual_price', lambda x: x[df_payment.loc[x.index, 'ph_status'] == 'DONE'].sum()),
    환불건수=('ph_status', lambda x: (x == 'CANCELED').sum()),
    환불총액=('actual_price', lambda x: x[df_payment.loc[x.index, 'ph_status'] == 'CANCELED'].sum())
).reset_index()

# 4. 데이터 정리 및 환불 비중 계산
total_promotion_stats['환불액비중(%)'] = (total_promotion_stats['환불총액'] / total_promotion_stats['구매총액'] * 100).fillna(0)

# 5. 시간 순서 및 유저 카테고리 순으로 정렬
total_promotion_stats = total_promotion_stats.sort_values(
    by=['결제시점_시작일', 'user_category', 'product_name']
)

# 6. 컬럼명 최종 정리 (가독성을 위해 기존 이름으로 복구)
total_promotion_stats.rename(columns={
    '결제시점_시작일': '프로모션_시작일',
    '결제시점_종료일': '프로모션_종료일',
    '결제시점_프로모션': '프로모션'
}, inplace=True)

# 컬럼 순서 조정
total_promotion_stats = total_promotion_stats[[
    'user_category', '프로모션_시작일', '프로모션_종료일', '프로모션', 'product_name',
    '구매건수', '구매총액', '환불건수', '환불총액', '환불액비중(%)'
]]

print("### [전체 유저] 결제 시점 프로모션 기준 구매 및 환불 현황 ###")
display(total_promotion_stats)

## 프로모션별 환불 요인분석

In [ ]:
import numpy as np

# 1. 분석 대상 필터링 (환불 유저만)
refund_categories = ['계약시작전환불', '계약시작후환불']
df_refund_users = df_payment[df_payment['user_category'].isin(refund_categories)].copy()

# 2. 유저별/시간순 정렬 후 직전 거래 정보 매핑
df_sorted = df_refund_users.sort_values(by=['phone_number', 'p_date', 'p_time'])

# 직전 거래의 정보들을 가져옵니다
df_sorted['이전_프로모션'] = df_sorted.groupby('phone_number')['프로모션'].shift(1)
df_sorted['이전_구매상품'] = df_sorted.groupby('phone_number')['product_name_clean'].shift(1)
df_sorted['이전_거래액'] = df_sorted.groupby('phone_number')['actual_price'].shift(1)
df_sorted['이전_상태'] = df_sorted.groupby('phone_number')['ph_status'].shift(1)

# 3. '취소(CANCELED) -> 신규결제(DONE)' 전환 케이스만 추출
migration_raw = df_sorted[
    (df_sorted['ph_status'] == 'DONE') &
    (df_sorted['이전_상태'] == 'CANCELED')
].copy()

# 4. 프로모션 이동 여부 카테고리 생성 (동일프로모션 vs 프로모션이동)
migration_raw['이동_구분'] = np.where(
    migration_raw['이전_프로모션'] == migration_raw['프로모션'],
    '동일프로모션',
    '프로모션이동'
)

# 5. 상세 항목별로 그룹화하여 집계
detailed_analysis = migration_raw.groupby(
    ['user_category', '이동_구분', '이전_프로모션', '프로모션', '이전_구매상품', 'product_name_clean']
).agg(
    이동유저수=('phone_number', 'nunique'),
    이전_거래액_총합=('이전_거래액', 'sum'),
    이후_거래액_총합=('actual_price', 'sum')
).reset_index()

# 6. 컬럼명 정리
detailed_analysis.columns = [
    '환불_카테고리', '이동_구분', '이전_프로모션', '신규_거래_프로모션',
    '이전_구매상품', '이후_구매상품',
    '이동유저수', '이전_거래액_총합', '이후_거래액_총합'
]

# 거래액 차이(증감) 컬럼 추가
detailed_analysis['거래액_변동'] = detailed_analysis['이후_거래액_총합'] - detailed_analysis['이전_거래액_총합']

# 7. 정렬 (환불_카테고리 -> 이동_구분 -> 이동유저수 순)
detailed_analysis = detailed_analysis.sort_values(
    by=['환불_카테고리', '이동_구분', '이동유저수'],
    ascending=[True, True, False]
)

print("### 환불 및 프로모션 이동 구분 상세 상세 분석 ###")
display(detailed_analysis)

In [ ]:
df_payment.head()

## 당일 환불케이스

In [ ]:
import pandas as pd

# 1. p_time이 이미 전체 일시를 포함하고 있으므로 이를 바로 타임스탬프로 변환
# (이미 데이터프레임에 있다면 astype(str)은 생략 가능하지만 안전을 위해 추가)
df_payment['full_timestamp'] = pd.to_datetime(df_payment['p_time'])

# 2. 동일한 transaction_id 내에서 DONE과 CANCELED 행 분리
done_df = df_payment[df_payment['ph_status'] == 'DONE'][['transaction_id', 'full_timestamp', '프로모션_결제시점']]
done_df.columns = ['transaction_id', 'done_time', '프로모션']

cancel_df = df_payment[df_payment['ph_status'] == 'CANCELED'][['transaction_id', 'full_timestamp']]
cancel_df.columns = ['transaction_id', 'cancel_time']

# 3. transaction_id 기준으로 병합
refund_gap_df = pd.merge(done_df, cancel_df, on='transaction_id')

# 4. 시간 차이 계산 및 시간대(Hour) 추출
refund_gap_df['time_diff'] = refund_gap_df['cancel_time'] - refund_gap_df['done_time']
refund_gap_df['diff_hours_float'] = refund_gap_df['time_diff'].dt.total_seconds() / 3600

# 5. 24시간 이내 환불 건만 필터링
quick_refunds = refund_gap_df[(refund_gap_df['diff_hours_float'] >= 0) & (refund_gap_df['diff_hours_float'] < 24)].copy()
quick_refunds['refund_hour_slot'] = quick_refunds['diff_hours_float'].astype(int)

# 6. 시간대별(0~23시) 환불 건수 집계
hourly_stats = quick_refunds.groupby('refund_hour_slot').size().reset_index(name='환불건수')

# 0~23시까지 빈 시간대 채우기
all_hours = pd.DataFrame({'refund_hour_slot': range(24)})
hourly_stats = pd.merge(all_hours, hourly_stats, on='refund_hour_slot', how='left').fillna(0)
hourly_stats['환불건수'] = hourly_stats['환불건수'].astype(int)

print("### 결제 후 24시간 이내 시간대별 환불 성과 ###")
display(hourly_stats)

In [ ]:
hourly_stats['환불건수'].sum()

In [ ]:
df_payment.head()

In [ ]:
import pandas as pd

def check_after_refund(group):
    # 1. 해당 유저의 환불(CANCELED 또는 WITHDRAW) 건들의 ID 추출
    refund_ids = group[group['status'].isin(['CANCELED', 'WITHDRAW'])]['transaction_id']

    if refund_ids.empty:
        return "환불이력없음"

    # [기준점] 가장 먼저 발생한 환불의 ID (가장 작은 값)
    first_refund_id = refund_ids.min()

    # 2. 첫 환불 ID보다 큰(즉, 이후에 발생한) 모든 트랜잭션 필터링
    after_refund_rows = group[group['transaction_id'] > first_refund_id]

    if after_refund_rows.empty:
        return "환불이후이탈"

    # 3. 이후 트랜잭션 중 status가 CANCELED나 WITHDRAW가 아닌 '정상 결제'가 있는지 확인
    actual_purchase = after_refund_rows[~after_refund_rows['status'].isin(['CANCELED', 'WITHDRAW'])]

    if not actual_purchase.empty:
        return "환불이후구매"
    else:
        return "환불이후이탈"

# [수정] include_groups=False 추가하여 경고 해결
reacquisition_results = df_payment_canceled.groupby('phone_number').apply(check_after_refund, include_groups=False)

# [수정] 변수명을 reacquisition_results로 통일하여 NameError 해결
df_payment_canceled['환불이후구매_여부'] = df_payment_canceled['phone_number'].map(reacquisition_results)

# 결과 확인
print(df_payment_canceled.drop_duplicates('phone_number')['환불이후구매_여부'].value_counts())

In [ ]:
df_payment_canceled.groupby('ph_status')['transaction_id'].nunique()

In [ ]:
# 1. 유저별 고유 결제(트랜잭션) 개수 확인
user_txn_counts = df_payment_canceled.groupby('phone_number')['transaction_id'].nunique()

# 2. 1개인 경우와 그 이상인 경우로 그룹핑 레이블 생성
# 1개면 '단건(이탈)', 2개 이상이면 '다건(재시도)'
user_groups = user_txn_counts.apply(lambda x: '1건 (단판이탈)' if x == 1 else '2건 이상 (재시도/복귀)')

# 3. 최종 그룹바이 집계 (유저 수 카운트)
cohort_result = user_groups.value_counts().reset_index()
cohort_result.head()


In [ ]:
df_payment_cancelContract = df_payment_canceled[df_payment_canceled['status'].str.contains('WITHDRAW|CANCELED', na=False)]
df_payment_cancelContract.sort_values('transaction_id')

In [ ]:
df_payment_cancelContract.groupby('ph_status')['transaction_id'].count()

In [ ]:
import pandas as pd

# [준비] 날짜 형식 강제 변환 (시간 정보가 꼬이지 않게 날짜만 추출)
df_payment_cancelContract['p_date'] = pd.to_datetime(df_payment_cancelContract['p_date']).dt.normalize()

# 1. 중복 데이터 제거
# 동일한 ID에 동일한 상태가 여러 번 찍힌 경우(이미지의 123, 124번 같은 상황)를 방지합니다.
df_clean = df_payment_cancelContract.drop_duplicates(subset=['transaction_id', 'ph_status'], keep='first')

# 2. 결제일(DONE)과 취소일(CANCELED) 분리 추출
# groupby 후 unstack을 쓰면 pivot_table보다 에러에 강합니다.
date_map = df_clean.groupby(['transaction_id', 'ph_status'])['p_date'].first().unstack()

# 3. 소요 일수 계산
# 만약 취소일이 결제일보다 앞서는 데이터 오류가 있다면 0으로 처리하도록 clip(lower=0) 추가
if 'DONE' in date_map.columns and 'CANCELED' in date_map.columns:
    date_map['duration'] = (date_map['CANCELED'] - date_map['DONE']).dt.days

    # 음수값이 나오는 경우(데이터 기록 오류)를 위해 0일로 보정 (선택 사항)
    # date_map['duration'] = date_map['duration'].clip(lower=0)

    # 4. 원본 데이터프레임에 매핑
    # 기존에 map이 안 되었던 이유는 transaction_id의 데이터 타입(int/float) 문제일 수 있으므로 타입을 맞춥니다.
    df_payment_cancelContract['cancel_duration'] = df_payment_cancelContract['transaction_id'].map(date_map['duration'])

# 결과 확인
df_payment_cancelContract.head()

In [ ]:
import re

def clean_product_name(name):
    if not isinstance(name, str):
        return name

    # 1. 기본 명칭 통일 (무제한 -> 무제한 패스, 주말&야간 -> 주말&야간 패스)
    # 이미 '패스'가 포함되어 있으면 중복해서 붙이지 않음
    if '무제한' in name and '패스' not in name:
        name = name.replace('무제한', '무제한 패스')
    if '주말&야간' in name and '패스' not in name:
        name = name.replace('주말&야간', '주말&야간 패스')

    # 2. 산술 연산 처리 (6x2 -> 12, 6+6 -> 12, 12+1 -> 13)
    # 곱셈 패턴 처리 (숫자 x 숫자)
    mult_match = re.search(r'(\d+)\s*[xX]\s*(\d+)', name)
    if mult_match:
        calc_val = int(mult_match.group(1)) * int(mult_match.group(2))
        name = re.sub(r'\d+\s*[xX]\s*\d+', str(calc_val), name)

    # 덧셈 패턴 처리 (숫자 + 숫자)
    plus_match = re.search(r'(\d+)\s*\+\s*(\d+)', name)
    if plus_match:
        calc_val = int(plus_match.group(1)) + int(plus_match.group(2))
        name = re.sub(r'\d+\s*\+\s*\d+', str(calc_val), name)

    # 3. 기타 불필요한 기호 및 공백 정리
    name = name.replace('+', ' ') # 숫자가 아닌 곳에 붙은 + 제거
    name = re.sub(r'\s+', ' ', name) # 중복 공백을 하나로 합침

    return name.strip()

# 데이터프레임 적용 예시
df_payment_cancelContract['product_name_clean'] = df_payment_cancelContract['product_name'].apply(clean_product_name)

In [ ]:
# cancel_duration 컬럼에서 0보다 작은 값은 모두 0으로 변경
df_payment_cancelContract['cancel_duration'] = df_payment_cancelContract['cancel_duration'].clip(lower=0)

In [ ]:
df_payment_cancelContract.head()

#계약시작후 환불(TERMINATION)

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_payment_termination = pd.read_sql("""
                        SELECT

                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS p_date,
                        TO_CHAR(ph.requested_at AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH:MM:SS') AS p_time,
                        c.contract_uid transaction_id,
                        c.client_uid uid,
                        c.status,
                        ph.payment_status AS ph_status,
                        c.product_name,
                        pp.name product_period,
                        TO_CHAR(c.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS start_date,
                        TO_CHAR(c.initial_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS initial_end_date,
                        TO_CHAR(c.actual_end_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS actual_end_date,
                        c.actual_price,
                        c.is_migrated,
                        u.phone_number
                        FROM contract c
                        LEFT JOIN payment p
                        ON c.contract_uid = p.contract_payment_uid
                        LEFT JOIN price_policy pp
                        ON c.price_policy_uid = pp.price_policy_uid
                        LEFT JOIN client u
                        ON c.client_uid = u.client_uid
                        LEFT JOIN payment_history ph
                        ON contract_uid = ph.payment_uid
                        where actual_price > 0
                        and c.status = 'TERMINATION'
                        and ph.payment_status = 'DONE'
                        ORDER BY ph.requested_at ASC;

                    """, conn)
    conn.close()
df_payment_termination['product_name'] = df_payment_termination['product_name'] + ' ' + df_payment_termination['product_period'].astype(str)
df_payment_termination = df_payment_termination.drop(columns=['product_period'])
df_payment_termination.tail()

In [ ]:
df_payment_termination.groupby('ph_status')['transaction_id'].count()

In [ ]:
import re

def clean_product_name(name):
    if not isinstance(name, str):
        return name

    # 1. 기본 명칭 통일 (무제한 -> 무제한 패스, 주말&야간 -> 주말&야간 패스)
    # 이미 '패스'가 포함되어 있으면 중복해서 붙이지 않음
    if '무제한' in name and '패스' not in name:
        name = name.replace('무제한', '무제한 패스')
    if '주말&야간' in name and '패스' not in name:
        name = name.replace('주말&야간', '주말&야간 패스')

    # 2. 산술 연산 처리 (6x2 -> 12, 6+6 -> 12, 12+1 -> 13)
    # 곱셈 패턴 처리 (숫자 x 숫자)
    mult_match = re.search(r'(\d+)\s*[xX]\s*(\d+)', name)
    if mult_match:
        calc_val = int(mult_match.group(1)) * int(mult_match.group(2))
        name = re.sub(r'\d+\s*[xX]\s*\d+', str(calc_val), name)

    # 덧셈 패턴 처리 (숫자 + 숫자)
    plus_match = re.search(r'(\d+)\s*\+\s*(\d+)', name)
    if plus_match:
        calc_val = int(plus_match.group(1)) + int(plus_match.group(2))
        name = re.sub(r'\d+\s*\+\s*\d+', str(calc_val), name)

    # 3. 기타 불필요한 기호 및 공백 정리
    name = name.replace('+', ' ') # 숫자가 아닌 곳에 붙은 + 제거
    name = re.sub(r'\s+', ' ', name) # 중복 공백을 하나로 합침

    return name.strip()

# 데이터프레임 적용 예시
df_payment_termination['product_name_clean'] = df_payment_termination['product_name'].apply(clean_product_name)
df_payment_termination.head()

In [ ]:
df_payment_termination.info()

In [ ]:
import pandas as pd

# 0. 오늘 날짜 설정 (시간 제외하고 날짜만)
today = pd.Timestamp.now().normalize()

# 1. 대상 컬럼들을 날짜 타입으로 변환
date_cols = ['p_date', 'actual_end_date', 'initial_end_date', 'start_date']
for col in date_cols:
    df_payment_termination[col] = pd.to_datetime(df_payment_termination[col], errors='coerce')

# 2. 특정 시점 이후 데이터만 필터링
df_payment_termination = df_payment_termination[df_payment_termination['p_date'] >= '2025-06-11'].copy()

# 3. 계약기간 계산을 위한 '참조 종료일' 설정
# initial_end_date가 오늘보다 미래면 오늘 날짜를 사용, 아니면 그대로 사용
ref_end_date = df_payment_termination['initial_end_date'].where(
    df_payment_termination['initial_end_date'] <= today, today
)

# 4. 기간 계산
# 환불소요기간: 실제 해지일까지 이용한 기간
df_payment_termination['환불소요기간'] = (df_payment_termination['actual_end_date'] - df_payment_termination['start_date']).dt.days

# 계약기간: (참조 종료일 - 시작일) + 1
df_payment_termination['계약기간'] = (ref_end_date - df_payment_termination['start_date']).dt.days + 1

# 5. '개월' 단위 상품만 필터링
df_payment_termination = df_payment_termination[df_payment_termination['product_name'].str.contains('개월', na=False)].copy()

# 6. 결측치 처리 및 정수형(int) 변환
for col in ['환불소요기간', '계약기간']:
    df_payment_termination[col] = df_payment_termination[col].fillna(0).astype(int)

# 최종 결과 확인
df_payment_termination.head()

In [ ]:
df_payment_termination.info()

In [ ]:


# 1. 구간(bins)과 이름(labels) 재설정
# 30일 이하: 10일 단위 / 100일 이상: 큼직하게 / 최대 200일 이상
bins = [0, 10, 20, 30, 60, 100, 200, np.inf]
labels = [
    '10일 이하',
    '11~20일',
    '21~30일',
    '31~60일',
    '61~100일',
    '101~200일',
    '201일 이상'
]

# 2. pd.cut 적용
df_payment_termination['계약구간'] = pd.cut(
    df_payment_termination['계약기간'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# 3. 결과 확인 (전체 데이터프레임 형태)
df_payment_termination.head()

In [ ]:
import pandas as pd
import numpy as np

# 1. 소진율 계산 (0~100 사이의 퍼센트값)
# 계약기간이 0인 경우 에러 방지를 위해 fillna(0) 처리
df_payment_termination['소진율'] = (df_payment_termination['환불소요기간'] / df_payment_termination['계약기간'] * 100).fillna(0)

# 100%를 초과하는 데이터가 혹시 있다면 100으로 고정 (데이터 정제)
df_payment_termination['소진율'] = df_payment_termination['소진율'].clip(upper=100)

# 2. 소진율 구간 설정 (10% 단위)
# 0~10, 10~20, ..., 90~100
bins_rate = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels_rate = [
    '10% 이하', '11~20%', '21~30%', '31~40%', '41~50%',
    '51~60%', '61~70%', '71~80%', '81~90%', '91~100%'
]

# 3. pd.cut 적용
df_payment_termination['소진율구간'] = pd.cut(
    df_payment_termination['소진율'],
    bins=bins_rate,
    labels=labels_rate,
    include_lowest=True
)

# 최종 결과 확인 (구글 시트 업로드 전 데이터프레임 형태)
df_payment_termination.head()

In [ ]:
import pandas as pd
import numpy as np

# 1. 환불소요기간 구간 설정
# 0일(당일), 1~10일, 11~20일, 21~30일, 31~60일, 61~100일, 101일 이상
bins_usage = [-1, 0, 10, 20, 30, 60, 100, np.inf]
labels_usage = [
    '0일 (당일취소)',
    '1~10일',
    '11~20일',
    '21~30일',
    '31~60일',
    '61~100일',
    '101일 이상'
]

# 2. pd.cut 적용
df_payment_termination['환불소요구간'] = pd.cut(
    df_payment_termination['환불소요기간'],
    bins=bins_usage,
    labels=labels_usage
)

# 3. 엑셀 복사용 요약 출력
summary_usage = df_payment_termination['환불소요구간'].value_counts().sort_index().reset_index()
summary_usage.columns = ['환불소요구간', '유저수']

print("환불소요구간\t유저수")
for _, row in summary_usage.iterrows():
    print(f"{row['환불소요구간']}\t{row['유저수']}")

## 체크인데이터

In [ ]:

# 2. SSH Tunnel 연결 및 DB 접속
with SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_username,
    ssh_pkey=pem_path,
    remote_bind_address=(db_host, db_port),
    local_bind_address=('localhost', 5433)
) as tunnel:

    # 3. DB 연결 (로컬 포트를 통해)
    conn = psycopg2.connect(
        host='localhost',
        port=5433,
        database=db_name,
        user=db_user,
        password=db_password
    )

    # 4. 쿼리 실행
    df_check_in = pd.read_sql("""
                               SELECT
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD') AS check_in_date,
                                        TO_CHAR(ch.start_time AT TIME ZONE 'UTC' AT TIME ZONE 'KST', 'YYYY-MM-DD HH24:MI:SS') AS check_in_Time,
                                        ch.check_in_history_uid,
                                        ci.contract_participant_uid,
                                        c.contract_uid,
                                        ch.branch_uid,
                                        b.sub_type,
                                        b.display_name
                                FROM check_in ci
                                left join check_in_history ch
                                on ci.check_in_uid = ch.check_in_uid
                                left join branch b
                                on ch.branch_uid  = b.branch_uid
                                left join contract_participant cp
                                on cp.contract_participant_uid = ci.contract_participant_uid
                                left join contract c
                                on c.contract_uid = cp.contract_uid
                                LEFT JOIN price_policy pp
                                ON c.price_policy_uid = pp.price_policy_uid
                                where c.actual_price > 0

                    """, conn)
    conn.close()
df_check_in.head()

In [ ]:
df_payment_termination.head()

## 날짜 정렬

In [ ]:
import pandas as pd

# 1. 데이터 로드 (파일 경로를 확인해 주세요)
# df_payment_termination = pd.read_csv('df_payment_termination.csv')

# 2. p_date 컬럼을 날짜 형식으로 변환
df_payment_termination['p_date'] = pd.to_datetime(df_payment_termination['p_date'])

# 3. 2026-04-17 이전의 데이터만 필터링 (2026-04-17 포함 이후 날짜 삭제)
df_filtered = df_payment_termination[df_payment_termination['p_date'] < '2026-04-17']
df_check_in = df_check_in[df_check_in['check_in_date'] < '2026-04-17']
# 4. 결과 확인 및 저장
df_filtered.head()

In [ ]:
df_merged = pd.merge(df_filtered, df_check_in,
                     left_on='transaction_id',
                     right_on='contract_uid',
                     how='left')

In [ ]:
df_merged.head()

In [ ]:
df_merged.info()

In [ ]:
df_payment_termination.info()

In [ ]:
df_merged['체크인횟수'] = df_merged.groupby('transaction_id')['check_in_history_uid'].transform('count')
df_merged.head()

In [ ]:
# 1. datetime 형식으로 변환 (필수)
df_merged['check_in_time'] = pd.to_datetime(df_merged['check_in_time'])

# 2. 시간(hour)만 숫자로 추출
df_merged['체크인시간'] = df_merged['check_in_time'].dt.hour
df_merged.head()

In [ ]:
# 1. 대상 유저의 전화번호(phone_number) 추출
# detailed_analysis나 migration_raw에서 해당 조건의 유저 리스트를 뽑습니다.
target_product = '주말&야간 패스 12개월'

target_phone_numbers = migration_raw[
    (migration_raw['user_category'] == '계약시작전환불') &
    (migration_raw['이동_구분'] == '프로모션이동') &
    (migration_raw['이전_구매상품'] == target_product) &
    (migration_raw['product_name_clean'] == target_product)
]['phone_number'].unique()

# 2. 전체 df_payment에서 해당 유저들의 모든 거래 이력 필터링
raw_data_view = df_payment[df_payment['phone_number'].isin(target_phone_numbers)].copy()

# 3. 유저별/시간순으로 정렬하여 '취소 -> 재결제' 흐름 확인
raw_data_view = raw_data_view.sort_values(by=['phone_number', 'p_date', 'p_time'])

# 4. 분석에 핵심적인 컬럼 위주로 출력
display_columns = [
    'phone_number', 'p_date', 'ph_status', '프로모션', '프로모션_결제시점',
    'product_name_clean', 'actual_price', 'user_category'
]

print(f"### [Raw Data] {target_product} 프로모션 갈아타기 유저 상세 이력 ###")
display(raw_data_view[display_columns])